# AIAvatarLearningApp - Train the Stage 2 feedback model in Colab

Two paths through this notebook:

- **Recommended (local demo)**: cells 1-7. Trains the model, converts the LoRA **adapter** to a small GGUF (~90-100 MB), saves to Drive. Then download to your laptop and run via Ollama on top of the base `phi3.5`. Fully offline demo.
- **Alternative (Colab-hosted)**: cells 1-5, then 8. Serves the full merged model from Colab via ngrok. Requires internet + an active Colab session during your demo.

**Before you start:** Runtime -> Change runtime type -> GPU -> A100 (or T4 if A100 isn't available). Save this notebook to Drive so it survives disconnects.

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi

## 2. Clone the repo

If the repo is private, swap the line for:
`!git clone https://<YOUR_GITHUB_TOKEN>@github.com/YuvalHaski/AIAvatarLearningApp.git`

In [ ]:
!git clone https://github.com/YuvalHaski/AIAvatarLearningApp.git
%cd AIAvatarLearningApp

## 3. Install training dependencies (~3 min)

In [ ]:
!pip install -q -r training/requirements.txt

## 4. Mount Google Drive for persistent output

Critical - Colab sessions disconnect, but anything written to Drive survives. The trained model (~7 GB) and the final GGUF (~2 GB) will both be saved there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/aiavatar_model

## 5. Train (~30-90 min depending on GPU)

Watch the eval loss decrease across the 3 epochs. Final output: `/content/drive/MyDrive/aiavatar_model/merged/`. Keep this browser tab visible while it runs.

In [ ]:
!python training/train.py \
    --data training/data \
    --out /content/drive/MyDrive/aiavatar_model

## 6. Convert the LoRA adapter to GGUF (recommended)

Converts **only the LoRA adapter** (`adapter/`) to a small (~90-100 MB) GGUF. You run this on Ollama on top of the base `phi3.5`, so the download stays tiny. This is the proven path you shipped last time.

The cell below first clones llama.cpp and installs its Python deps (needed by the converter), then runs `convert_lora_to_gguf.py`. It pulls only the base model's **config** from HF, not the full weights, so it's fast.

> Alternative (merged -> quantized GGUF): the tied-weights bug that previously produced a *garbage* merged GGUF is fixed in `train.py` now (the `_tied_weights_keys` clearing in the merge step), but it yields a ~2.5 GB file and you'd serve it with `FROM ./aiavatar-feedback-q4_k_m.gguf`. Only fall back to it if the adapter-on-base path misbehaves.

In [ ]:
# Clone llama.cpp and install its Python deps - convert_lora_to_gguf.py needs them.
# No compile is needed for the adapter path. (The merged-GGUF fallback additionally
# needs: cmake -S /content/llama.cpp -B /content/llama.cpp/build -DLLAMA_CURL=OFF
# -DGGML_CUDA=OFF && cmake --build /content/llama.cpp/build --target llama-quantize)
!git clone --depth=1 https://github.com/ggerganov/llama.cpp.git /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt

In [ ]:
# Convert the trained LoRA adapter to a small GGUF (~90-100 MB).
# --base-model-id pulls ONLY the base model's config from HF (not the weights),
# so this is fast. The adapter was saved by train.py to <out>/adapter.
!python /content/llama.cpp/convert_lora_to_gguf.py \
    /content/drive/MyDrive/aiavatar_model/adapter \
    --base-model-id microsoft/Phi-3.5-mini-instruct \
    --outtype f16 \
    --outfile /content/drive/MyDrive/aiavatar_model/aiavatar-adapter.gguf

In [ ]:
# Sanity check: a healthy adapter GGUF is ~90-100 MB. If it's only a few KB the
# conversion silently dropped the LoRA tensors (check target_modules in train.py).
import os
p = "/content/drive/MyDrive/aiavatar_model/aiavatar-adapter.gguf"
mb = os.path.getsize(p) / 1e6
print(f"{p}\n{mb:.1f} MB")
assert mb > 10, "adapter GGUF is suspiciously small - conversion likely dropped tensors"

In [ ]:
# Verify the GGUF landed in Drive.
!ls -lh /content/drive/MyDrive/aiavatar_model/*.gguf

## 7. Download the adapter GGUF to your laptop and serve via Ollama

Do these steps on your **Windows laptop**, not in Colab:

1. Open Google Drive, go to `MyDrive/aiavatar_model/`, and download `aiavatar-adapter.gguf` (~90-100 MB).
2. Move it into your project at `AIAvatarLearningApp/models/aiavatar-adapter.gguf`, **overwriting** the old one.
3. Your existing `models/Modelfile` already points at it, so no edit is needed:
   ```
   FROM phi3.5
   ADAPTER ./aiavatar-adapter.gguf
   PARAMETER temperature 0.3
   ```
   (If you've never pulled the base, run `ollama pull phi3.5` once.)
4. In PowerShell, from the `models/` folder:
   ```powershell
   ollama rm feedback-model        # drop the model built from the OLD adapter
   ollama create feedback-model -f Modelfile
   ollama serve                    # if it isn't already running as a service
   ```
5. In your backend `.env`, set:
   ```
   FEEDBACK_MODEL_URL=http://localhost:11434/v1
   FEEDBACK_MODEL_NAME=feedback-model
   ```
6. Restart your FastAPI backend. Feedback now comes from your retrained adapter running fully on your laptop.

If you have an NVIDIA GPU, Ollama uses it automatically. Otherwise it runs on CPU - usable but slower (~3-8s per response).

## 8. (Alternative) Serve from Colab via ngrok

Only use this if you can't run Ollama locally for some reason. Tied to the Colab session - your laptop must have internet during the demo, and Colab can disconnect.

Sign up for a free ngrok account at https://dashboard.ngrok.com/signup, copy your auth token, and paste it below.

In [ ]:
!pip install -q vllm pyngrok

NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"
!ngrok config add-authtoken {NGROK_TOKEN}

In [ ]:
import subprocess, time

proc = subprocess.Popen([
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "/content/drive/MyDrive/aiavatar_model/merged",
    "--served-model-name", "feedback-model",
    "--port", "8000",
])

print("waiting 60s for vLLM to come up...")
time.sleep(60)

from pyngrok import ngrok
tunnel = ngrok.connect(8000)
print()
print("Put this in your local .env:")
print(f"  FEEDBACK_MODEL_URL={tunnel.public_url}/v1")

## 9. (Optional) Evaluate the trained model on the val set

Runs the faithfulness checks from `training/evaluate.py` against the running vLLM server. Only works if you ran section 8 first (Ollama runs on your laptop, not in Colab).

In [ ]:
!python training/evaluate.py \
    --endpoint http://localhost:8000/v1 \
    --model-name feedback-model \
    --val training/data/val.jsonl